# ENZERA - 03 Feature experiments: gap handling and weekly patterns

Two weaknesses found in the EDA (`01_eda.ipynb`):

1. **Long gaps are interpolated into flat lines.** 22.3% of customers have no reading before 2016. `clean()` fills
   that whole period with a copy of their first value, which distorts variability and trend features.
2. **The weekly cycle is ignored.** Zheng et al. (2018) show that weekly periodicity carries theft signal, but only
   `weekend_ratio` touches it.

**Variants** (same XGBoost settings, split, validation-tuned threshold and `evaluate()` as `ml/train.py`):

| Variant | Features |
|---|---|
| A. current | the 14 features of `ml/features.py` |
| B. gap-aware | the 14 features computed **only on the observed span** of each customer (no filling before the first or after the last reading) + `active_span_ratio` |
| C. weekly | A + `weekly_autocorr` (correlation of usage with usage 7 days earlier) + `dow_strength` (how much usage depends on the day of the week) |
| D. gap-aware + weekly | B + the two weekly features |

**Decision rule (fixed before running, same as notebook 02):** adopt a variant only if it improves **test PR-AUC by ≥ 0.01
AND test top-10% recall by ≥ 0.01**, and is not worse on validation PR-AUC. If several qualify, take the best test PR-AUC.

> **Provenance.** Variant *A* here is the original 14-feature `compute_features` (git commit `2ac5581`). After this experiment, variant *D* was adopted into `ml/features.py`, so re-running as-is would make A identical to D. To reproduce: `git checkout 2ac5581 -- ml/features.py reports/metrics.json` (temporarily).

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from ml.features import EPS, FEATURE_NAMES, compute_features, split_wide, _longest_run
from ml.train import SEED, best_f1_threshold, evaluate
import matplotlib.pyplot as plt
%matplotlib inline

ids, labels, dates, values = split_wide(pd.read_csv(ROOT / "data/raw/data.csv"))
y = labels
idx = np.arange(len(ids))
tr, rest = train_test_split(idx, test_size=0.30, stratify=labels, random_state=SEED)
te, demo = train_test_split(rest, test_size=0.50, stratify=labels[rest], random_state=SEED)
tr_fit, val = train_test_split(tr, test_size=0.15, stratify=labels[tr], random_state=SEED)
print(len(tr_fit), len(val), len(te))

25211 4449 6356


## New feature code
`clean_inside` is `clean()` with one change: interpolation fills gaps **between** readings only (`limit_area="inside"`).
Days before the first or after the last reading stay missing, and every feature uses NaN-aware statistics on the days
that exist.

In [2]:
def clean_inside(values):
    x = pd.DataFrame(values).interpolate(axis=1, limit_area="inside").to_numpy(np.float32)
    x = np.clip(x, 0, None)
    with np.errstate(all="ignore"):
        cap = np.nanmean(x, axis=1, keepdims=True) + 3 * np.nanstd(x, axis=1, keepdims=True)
    return np.where(np.isnan(x), np.nan, np.minimum(x, cap))


def nan_monthly_means(x, dates):
    months = dates.to_period("M").astype(str).to_numpy()
    out = []
    for mth in pd.unique(months):
        with np.errstate(all="ignore"):
            out.append(np.nanmean(x[:, months == mth], axis=1))
    return np.column_stack(out)


def features_gap_aware(values_raw, dates):
    """The same 14 features, computed only on each customer's observed span."""
    missing_ratio = np.isnan(values_raw).mean(axis=1)
    x = clean_inside(values_raw)
    obs = ~np.isnan(x)
    n_obs = np.maximum(obs.sum(axis=1), 1)
    with np.errstate(all="ignore"):
        mean = np.nanmean(x, axis=1); std = np.nanstd(x, axis=1)
        m = nan_monthly_means(x, dates)
        mv = ~np.isnan(m); t = np.arange(m.shape[1], dtype=float)
        w = mv.sum(axis=1)
        tm = (mv * t).sum(axis=1) / np.maximum(w, 1)
        mm = np.nanmean(m, axis=1)
        dt = np.where(mv, t - tm[:, None], 0.0); dm = np.where(mv, m - mm[:, None], 0.0)
        slope = (dt * dm).sum(axis=1) / np.maximum((dt ** 2).sum(axis=1), EPS)
        drops = (m[:, :-1] - m[:, 1:]) / (m[:, :-1] + EPS)
        max_drop = np.clip(np.nan_to_num(np.nanmax(drops, axis=1), nan=0.0), 0, 1)
        recent = 90
        rv = np.nanmean(x[:, -recent:], axis=1) / (np.nanmean(x[:, :-recent], axis=1) + EPS)
        weekend = dates.dayofweek.to_numpy() >= 5
        wr = np.nanmean(x[:, weekend], axis=1) / (np.nanmean(x[:, ~weekend], axis=1) + EPS)
    zero = np.where(obs, x <= EPS, False)
    first = np.where(obs.any(axis=1), obs.argmax(axis=1), x.shape[1])
    last = np.where(obs.any(axis=1), x.shape[1] - 1 - obs[:, ::-1].argmax(axis=1), -1)
    f = pd.DataFrame({
        "mean_daily": mean, "std_daily": std, "cv_daily": std / (mean + EPS), "max_daily": np.nanmax(np.where(obs, x, -np.inf), axis=1),
        "missing_ratio": missing_ratio, "zero_ratio": zero.sum(axis=1) / n_obs,
        "longest_zero_run": [_longest_run(r) for r in zero],
        "low_day_ratio": np.where(obs, x < 0.2 * mean[:, None], False).sum(axis=1) / n_obs,
        "spike_ratio": np.where(obs, x > (mean + 2 * std)[:, None], False).sum(axis=1) / n_obs,
        "trend_slope": slope / (mean + EPS), "max_monthly_drop": max_drop,
        "monthly_cv": np.nanstd(m, axis=1) / (mm + EPS),
        "recent_vs_past": np.clip(np.nan_to_num(rv, nan=1.0), 0, 10),
        "weekend_ratio": np.clip(np.nan_to_num(wr, nan=1.0), 0, 10),
        "active_span_ratio": np.clip(last - first + 1, 0, None) / x.shape[1],
    })
    return f.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)


def weekly_features(x, dates):
    """x: cleaned daily matrix (may contain NaN outside the observed span)."""
    with np.errstate(all="ignore"):
        d = x - np.nanmean(x, axis=1, keepdims=True)
        a, b = d[:, 7:], d[:, :-7]
        ok = ~np.isnan(a) & ~np.isnan(b)
        a0, b0 = np.where(ok, a, 0), np.where(ok, b, 0)
        ac = (a0 * b0).sum(axis=1) / np.sqrt((a0 ** 2).sum(axis=1) * (b0 ** 2).sum(axis=1))
        dow = dates.dayofweek.to_numpy()
        dow_means = np.column_stack([np.nanmean(x[:, dow == k], axis=1) for k in range(7)])
        strength = np.nanvar(dow_means, axis=1) / np.nanvar(x, axis=1)
    return pd.DataFrame({"weekly_autocorr": np.nan_to_num(ac, nan=0.0),
                         "dow_strength": np.clip(np.nan_to_num(strength, nan=0.0), 0, 1)}).astype(np.float32)

In [3]:
from ml.features import clean
F_A = compute_features(values, dates)
F_B = features_gap_aware(values, dates)
W_old = weekly_features(clean(values), dates)        # weekly features on the current cleaning
W_new = weekly_features(clean_inside(values), dates)  # weekly features on the gap-aware cleaning
variants = {
    "A. current (14)": F_A,
    "B. gap-aware (15)": F_B,
    "C. weekly (16)": pd.concat([F_A, W_old], axis=1),
    "D. gap-aware + weekly (17)": pd.concat([F_B, W_new], axis=1),
}
for k, v in variants.items():
    print(f"{k:28s} shape {v.shape}  any NaN: {bool(v.isna().any().any())}")

C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:5: RuntimeWarning: Mean of empty slice
  cap = np.nanmean(x, axis=1, keepdims=True) + 3 * np.nanstd(x, axis=1, keepdims=True)


D:\nirma-mtech-ds\aml lab\enzera\venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:25: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(x, axis=1); std = np.nanstd(x, axis=1)


C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:14: RuntimeWarning: Mean of empty slice
  out.append(np.nanmean(x[:, months == mth], axis=1))


C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:30: RuntimeWarning: Mean of empty slice
  mm = np.nanmean(m, axis=1)
C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:34: RuntimeWarning: All-NaN slice encountered
  max_drop = np.clip(np.nan_to_num(np.nanmax(drops, axis=1), nan=0.0), 0, 1)
C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:36: RuntimeWarning: Mean of empty slice
  rv = np.nanmean(x[:, -recent:], axis=1) / (np.nanmean(x[:, :-recent], axis=1) + EPS)


C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:38: RuntimeWarning: Mean of empty slice
  wr = np.nanmean(x[:, weekend], axis=1) / (np.nanmean(x[:, ~weekend], axis=1) + EPS)


... (3 outputs left out)

C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:60: RuntimeWarning: Mean of empty slice
  d = x - np.nanmean(x, axis=1, keepdims=True)


C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:66: RuntimeWarning: Mean of empty slice
  dow_means = np.column_stack([np.nanmean(x[:, dow == k], axis=1) for k in range(7)])


C:\Users\SAKSHI\AppData\Local\Temp\ipykernel_5044\2893271889.py:67: RuntimeWarning: Degrees of freedom <= 0 for slice.
  strength = np.nanvar(dow_means, axis=1) / np.nanvar(x, axis=1)


A. current (14)              shape (42372, 14)  any NaN: False
B. gap-aware (15)            shape (42372, 15)  any NaN: False
C. weekly (16)               shape (42372, 16)  any NaN: False
D. gap-aware + weekly (17)   shape (42372, 17)  any NaN: False


### Do the new features separate the classes? (train split only, single-feature ROC-AUC)

In [4]:
from sklearn.metrics import roc_auc_score
new = pd.concat([F_B[["active_span_ratio"]], W_new], axis=1)
print(pd.Series({c: roc_auc_score(y[tr], new[c].iloc[tr]) for c in new.columns}).round(3).to_string())
print("\nCurrent vs gap-aware, single-feature ROC-AUC on train for the most affected features:")
cmp = pd.DataFrame({"current": {c: roc_auc_score(y[tr], F_A[c].iloc[tr]) for c in ["std_daily", "trend_slope", "monthly_cv", "max_monthly_drop", "recent_vs_past"]},
                    "gap-aware": {c: roc_auc_score(y[tr], F_B[c].iloc[tr]) for c in ["std_daily", "trend_slope", "monthly_cv", "max_monthly_drop", "recent_vs_past"]}})
cmp.round(3)

active_span_ratio    0.483
weekly_autocorr      0.633
dow_strength         0.454

Current vs gap-aware, single-feature ROC-AUC on train for the most affected features:


,current,gap-aware
std_daily,0.705,0.710
trend_slope,0.612,0.619
monthly_cv,0.593,0.593
max_monthly_drop,0.539,0.537
recent_vs_past,0.613,0.623


## Train and evaluate each variant

In [5]:
neg, pos = np.bincount(y[tr_fit]); spw = round(neg / pos, 3)
PARAMS = {"n_estimators": 400, "max_depth": 5, "learning_rate": 0.05, "subsample": 0.8,
          "colsample_bytree": 0.8, "scale_pos_weight": spw}
rows, probs = [], {}
for name, F in variants.items():
    X = F.to_numpy()
    model = XGBClassifier(**PARAMS, eval_metric="aucpr", n_jobs=-1, random_state=SEED).fit(X[tr_fit], y[tr_fit])
    p_val, p_te = model.predict_proba(X[val])[:, 1], model.predict_proba(X[te])[:, 1]
    r = evaluate(y[te], p_te, best_f1_threshold(y[val], p_val))
    r["val_pr_auc"] = float(average_precision_score(y[val], p_val))
    rows.append({"variant": name, **r}); probs[name] = p_te
res = pd.DataFrame(rows).set_index("variant")
saved = json.loads((ROOT / "reports/metrics.json").read_text())["results"]["xgboost"]
assert abs(res.loc["A. current (14)", "pr_auc"] - saved["pr_auc"]) < 1e-6, "baseline must reproduce metrics.json"
table = res[["precision", "recall", "f1", "roc_auc", "pr_auc", "recall_at_top10", "val_pr_auc"]].round(3)
table.columns = ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "Theft caught in top 10%", "PR-AUC (validation)"]
table.to_csv(ROOT / "reports/feature_experiments.csv")
table

,Precision,Recall,F1,ROC-AUC,PR-AUC,Theft caught in top 10%,PR-AUC (validation)
variant,,,,,,,
A. current (14),0.363,0.366,0.365,0.788,0.332,0.405,0.338
B. gap-aware (15),0.425,0.363,0.391,0.795,0.366,0.424,0.374
C. weekly (16),0.391,0.319,0.351,0.788,0.329,0.398,0.334
D. gap-aware + weekly (17),0.394,0.394,0.394,0.798,0.376,0.435,0.379


## Decision

In [6]:
base = res.loc["A. current (14)"]
dec = pd.DataFrame({"d PR-AUC (test)": res["pr_auc"] - base["pr_auc"],
                    "d top10 (test)": res["recall_at_top10"] - base["recall_at_top10"],
                    "d PR-AUC (val)": res["val_pr_auc"] - base["val_pr_auc"]}).drop("A. current (14)")
dec["qualifies?"] = (dec["d PR-AUC (test)"] >= 0.01) & (dec["d top10 (test)"] >= 0.01) & (dec["d PR-AUC (val)"] >= 0)
print(dec.round(4).to_string())
q = dec[dec["qualifies?"]]
winner = q["d PR-AUC (test)"].idxmax() if len(q) else None
print("\nDECISION:", f"adopt {winner}" if winner else "keep the current 14 features")

                            d PR-AUC (test)  d top10 (test)  d PR-AUC (val)  qualifies?
variant                                                                                
B. gap-aware (15)                    0.0340          0.0184          0.0364        True
C. weekly (16)                      -0.0038         -0.0074         -0.0037       False
D. gap-aware + weekly (17)           0.0430          0.0295          0.0409        True

DECISION: adopt D. gap-aware + weekly (17)


### Paired bootstrap: is the best variant's gain real?

In [7]:
best = res.drop("A. current (14)")["pr_auc"].idxmax()
rng = np.random.default_rng(SEED); yt = y[te]; diffs = []
for _ in range(1000):
    b = rng.integers(0, len(yt), len(yt))
    diffs.append(average_precision_score(yt[b], probs[best][b]) - average_precision_score(yt[b], probs["A. current (14)"][b]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"{best}: PR-AUC {res.loc[best, 'pr_auc'] - base['pr_auc']:+.4f} vs current, 95% CI [{lo:+.4f}, {hi:+.4f}]")

D. gap-aware + weekly (17): PR-AUC +0.0430 vs current, 95% CI [+0.0168, +0.0670]


## Conclusion

_(pending)_